**Arquitectura de Software a Gran Escala, 2026i**</br>
Jeisson Andrés Vergara Vargas

# Verificación Arquitectónica — Subsistema de Notificación

## Descripción

En este laboratorio se explorará el **Subsistema de Notificación** (modelado en `subsystem_model.sos`) de la arquitectura COSO a través de una simulación utilizando el lenguaje de programación Python. Utilizando un modelo arquitectónico base, se han agregado funciones para soportar el patrón arquitectónico de **balanceo de carga**.

El subsistema gestiona el flujo de alertas desde sistemas externos (`data_adquisition_system`, `c4_system`) hacia diversos canales de notificación (SMS, radio CAP, push).

## Objetivo General

Desarrollar una comprensión profunda de la arquitectura de software a gran escala mediante el modelado, simulación y análisis del subsistema de notificación. Además, evidenciar cómo las tácticas arquitectónicas son útiles para soportar atributos de calidad como el rendimiento, la disponibilidad, etc.

## Objetivos Específicos

1. **Modelado**: Construir una representación abstracta de la arquitectura del subsistema de notificación.
2. **Instanciación**: Crear y configurar los componentes que conforman la arquitectura del sistema.
3. **Balanceo de carga**: Utilizar técnicas de balanceo de carga para soportar el tráfico en momentos en que los componentes no tienen la capacidad completa.
4. **Resultados**: Analizar los datos obtenidos para identificar comportamientos exitosos y fallidos, así como los componentes críticos que afectan al sistema. Analizar los resultados en cada una de las técnicas de balanceo de carga aplicadas.

# Pasos de Ejecución

1. Modelado
2. Simulación con gráfico / simulación sin gráfico
3. Ejecutar alguna de las secciones de arquitectura
   - Arquitectura original
   - Balanceador de carga básico
   - Múltiple balanceador de carga
   - Round Robin (RR)
   - Round Robin ponderado
4. Ejecutar la simulación

# Modelado

In [3]:
pip install networkx matplotlib

  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 89.0 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [20]:
import threading
import time
import random
import networkx as nx
import matplotlib.pyplot as plt
from collections import defaultdict

# ================================
# 1. Modeling
# ================================

class Component:
    """
    Base class for all architecture components.
    """
    def __init__(self, name, component_type, capacity=100):
        self.name = name
        self.component_type = component_type
        self.capacity = capacity
        self.lock = threading.Lock()
        self.current_load = 0

    def process_transaction(self, transaction):
        """
        Processes a transaction. Returns True if the transaction is processed successfully,
        or False if the component's capacity is exceeded.
        """
        with self.lock:
            if self.current_load >= self.capacity:
                return False  # Over capacity failure
            self.current_load += 1
        # Simulate processing time
        processing_time = random.uniform(0.01, 0.05)
        time.sleep(processing_time)
        with self.lock:
            self.current_load -= 1
        return True

class PresentationTier(Component):
    """
    Class for API Gateway components (Presentation Tier)
    """
    def __init__(self, name, capacity=100):
        super().__init__(name, "Presentation Tier", capacity)

class LogicTier(Component):
    """
    Class for microservice and logic components (Logic Tier)
    """
    def __init__(self, name, capacity=100):
        super().__init__(name, "Logic Tier", capacity)

class ExternalServiceTier(Component):
    """
    Class for external service components (External Service Tier)
    """
    def __init__(self, name, capacity=200):
        super().__init__(name, "External Service Tier", capacity)

class DataTier(Component):
    """
    Class for database and cache components (Data Tier)
    """
    def __init__(self, name, capacity=100):
        super().__init__(name, "Data Tier", capacity)

class Transaction:
    """
    Represents a transaction in the notification subsystem.
    """
    def __init__(self, source, transaction_type, channel=None):
        self.source = source  # Source system identifier (e.g., "data_adquisition_system")
        self.transaction_type = transaction_type  # "alert_trigger" or "status_query"
        self.channel = channel  # Notification channel (sms, radio, push) for alert triggers
        self.components_involved = []
        self.status = None  # "Success" or "Failed"

In [21]:
def represent_graph(graph, components):
    """
    Represents the architecture as a graph.
    """
    plt.figure(figsize=(14, 10))
    pos = nx.spring_layout(graph, k=0.5, iterations=50)

    # Assign colors based on component type
    color_map = {
        'Presentation Tier': 'lightblue',
        'Logic Tier': 'lightgreen',
        'External Service Tier': 'coral',
        'Data Tier': 'orange'
    }
    node_colors = [color_map[components[node].component_type] for node in graph.nodes()]

    nx.draw(
        graph,
        pos,
        with_labels=True,
        node_color=node_colors,
        node_size=5000,
        font_size=7,
        font_weight='bold',
        arrows=True
    )
    plt.title("Notification Subsystem Architecture Represented as a Graph")
    plt.show()

def visualize_metrics(metrics):
    """
    Visualizes the collected metrics using bar charts.
    """
    total = len(metrics)
    success = sum(1 for tx in metrics if tx.status == 'Success')
    failed = sum(1 for tx in metrics if tx.status == 'Failed')

    labels = ['Successful', 'Failed']
    counts = [success, failed]
    colors = ['green', 'red']

    plt.figure(figsize=(6, 6))
    plt.bar(labels, counts, color=colors)
    plt.title('Successful vs Failed Transactions')
    plt.xlabel('Status')
    plt.ylabel('Count')
    plt.show()

    type_metrics = defaultdict(lambda: {'Success': 0, 'Failed': 0})
    for tx in metrics:
        type_metrics[tx.transaction_type][tx.status] += 1

    types = list(type_metrics.keys())
    success_counts = [type_metrics[t]['Success'] for t in types]
    failed_counts = [type_metrics[t]['Failed'] for t in types]

    x = range(len(types))

    plt.figure(figsize=(12, 8))
    plt.bar(x, success_counts, width=0.4, label='Successful', color='green')
    plt.bar([p + 0.4 for p in x], failed_counts, width=0.4, label='Failed', color='red')
    plt.xlabel('Transaction Type')
    plt.ylabel('Count')
    plt.title('Transactions by Type')
    plt.xticks([p + 0.2 for p in x], [t.capitalize() for t in types], rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()

def report_metrics(metrics):
    """
    Displays the metrics collected during the simulation.
    """
    total = len(metrics)
    success = sum(1 for tx in metrics if tx.status == 'Success')
    failed = sum(1 for tx in metrics if tx.status == 'Failed')
    print(f"\n=== Simulation Results ===")
    print(f"Total Transactions: {total}")
    print(f"Successful Transactions: {success}")
    print(f"Failed Transactions: {failed}\n")

    type_metrics = defaultdict(lambda: {'Success': 0, 'Failed': 0})
    for tx in metrics:
        type_metrics[tx.transaction_type][tx.status] += 1

    print("Summary by Transaction Type:")
    for t_type, counts in type_metrics.items():
        print(f"  {t_type.capitalize()}: {counts['Success']} successful, {counts['Failed']} failed")

    component_metrics = defaultdict(lambda: {'Processed': 0, 'Failed': 0})
    for tx in metrics:
        for comp in tx.components_involved:
            component_metrics[comp]['Processed'] += 1
        if tx.status == 'Failed':
            for comp in tx.components_involved:
                component_metrics[comp]['Failed'] += 1

    print("\nSummary by Component:")
    for comp, counts in component_metrics.items():
        print(f"  {comp}: Processed {counts['Processed']}, Failed {counts['Failed']}")

    print("\nDetails of Failed Transactions:")
    for tx in metrics:
        if tx.status == 'Failed':
            if tx.transaction_type == 'alert_trigger':
                source = f"{tx.source} via {tx.channel}"
            else:
                source = tx.source
            print(f"Source: {source}, Type: {tx.transaction_type}, Components: {tx.components_involved}, Status: {tx.status}")

def run_simulation(components, metrics, alert_count=1200, status_count=800):
    """
    Runs the simulation of concurrent transactions.
    """
    channels = ["sms_service", "cap_radio_difussion", "notification_push"]

    threads = []
    for i in range(1, alert_count + 1):
        source = "data_adquisition_system" if i % 2 == 0 else "c4_system"
        channel = random.choice(channels)
        transaction = Transaction(source, 'alert_trigger', channel)
        thread = threading.Thread(target=simulate_transaction, args=(transaction, components, metrics))
        threads.append(thread)
        thread.start()

    for i in range(1, status_count + 1):
        transaction = Transaction("personnel_orchestration_system", 'status_query')
        thread = threading.Thread(target=simulate_transaction, args=(transaction, components, metrics))
        threads.append(thread)
        thread.start()

    for thread in threads:
        thread.join()


# Simulación con gráfico

En la fase de Graficación, se generan representaciones visuales de la arquitectura del sistema y de los resultados obtenidos durante la simulación. La visualización es una herramienta poderosa para analizar y comunicar la estructura del sistema, así como para interpretar los datos de rendimiento y detectar posibles cuellos de botella o puntos de fallo.

In [6]:
def run_notification_simulation():
    """
    Configura la arquitectura del subsistema de notificación, ejecuta la simulación de transacciones,
    reporta los resultados, visualiza las métricas y representa la arquitectura antes y después
    de inducir una falla.
    """
    # Setup architecture
    components, graph = setup_architecture()

    # Represent the initial architecture
    print("=== Initial Architecture Representation ===")
    represent_graph(graph, components)

    # Metrics collection
    metrics = []

    # Simulate transactions with initial capacity
    print("\n=== Simulating Transactions with Initial Architecture ===")
    run_simulation(components, metrics)
    report_metrics(metrics)
    visualize_metrics(metrics)

    # Induce a failure in the architecture
    induce_failure(components)

    # Clear previous metrics
    metrics.clear()

    # Simulate transactions after inducing failure
    print("\n=== Simulating Transactions with Failure Scenario ===")
    run_simulation(components, metrics)
    report_metrics(metrics)
    visualize_metrics(metrics)

    # Represent the architecture after failure
    print("\n=== Architecture Representation After Failure ===")
    represent_graph(graph, components)

# Simulación sin gráfico

In [22]:
def run_notification_simulation():
    """
    Configura la arquitectura del subsistema de notificación, ejecuta la simulación de transacciones,
    reporta los resultados, visualiza las métricas y representa la arquitectura antes y después
    de inducir una falla.
    """
    # Setup architecture
    components, graph = setup_architecture()

    # Metrics collection
    metrics = []

    # Simulate transactions with initial capacity
    print("\n=== Simulating Transactions with Initial Architecture ===")
    run_simulation(components, metrics)
    report_metrics(metrics)

    # Induce a failure in the architecture
    induce_failure(components)

    # Clear previous metrics
    metrics.clear()

    # Simulate transactions after inducing failure
    print("\n=== Simulating Transactions with Failure Scenario ===")
    run_simulation(components, metrics)
    report_metrics(metrics)


# Arquitectura original

En esta sección de código se encuentra el modelado de la arquitectura del subsistema de notificación básica, sin balanceo de carga.

Componentes del modelo (`subsystem_model.sos`):
- **Presentation Tier**: `api_gateway`
- **Logic Tier**: `data_validator`, `event_bus`, `alert_rules_engine_ms`, `notification_emitter`, `delivery_state_tracker`
- **External Service Tier**: `sms_service`, `cap_radio_difussion`, `notification_push`
- **Data Tier**: `alerts_events_db`, `delivery_state_db`, `alerts_events_cache`

Tipos de transacción:
- `alert_trigger`: Flujo completo de alerta (api_gateway → data_validator → event_bus → alert_rules_engine_ms → alerts_events_db → notification_emitter → servicio externo)
- `status_query`: Consulta de estado de entrega (api_gateway → event_bus → delivery_state_tracker → delivery_state_db)

In [23]:
def setup_architecture():
    """
    Sets up the notification subsystem architecture by instantiating components and defining their connections.
    Based on subsystem_model.sos
    """
    components = {}
    graph = nx.DiGraph()

    # Presentation Tier
    components['api_gateway'] = PresentationTier('api_gateway', capacity=200)

    # Logic Tier
    components['data_validator'] = LogicTier('data_validator', capacity=200)
    components['event_bus'] = LogicTier('event_bus', capacity=500)
    components['alert_rules_engine_ms'] = LogicTier('alert_rules_engine_ms', capacity=300)
    components['notification_emitter'] = LogicTier('notification_emitter', capacity=300)
    components['delivery_state_tracker'] = LogicTier('delivery_state_tracker', capacity=200)

    # External Service Tier
    components['sms_service'] = ExternalServiceTier('sms_service', capacity=500)
    components['cap_radio_difussion'] = ExternalServiceTier('cap_radio_difussion', capacity=300)
    components['notification_push'] = ExternalServiceTier('notification_push', capacity=400)

    # Data Tier
    components['alerts_events_db'] = DataTier('alerts_events_db', capacity=300)
    components['delivery_state_db'] = DataTier('delivery_state_db', capacity=300)
    components['alerts_events_cache'] = DataTier('alerts_events_cache', capacity=400)

    # Define connections between components (based on subsystem_model.sos connectors)
    graph.add_edge('api_gateway', 'data_validator')
    graph.add_edge('data_validator', 'event_bus')
    graph.add_edge('event_bus', 'alert_rules_engine_ms')
    graph.add_edge('event_bus', 'notification_emitter')
    graph.add_edge('event_bus', 'delivery_state_tracker')
    graph.add_edge('alert_rules_engine_ms', 'alerts_events_db')
    graph.add_edge('alert_rules_engine_ms', 'alerts_events_cache')
    graph.add_edge('notification_emitter', 'sms_service')
    graph.add_edge('notification_emitter', 'cap_radio_difussion')
    graph.add_edge('notification_emitter', 'notification_push')
    graph.add_edge('notification_emitter', 'delivery_state_tracker')
    graph.add_edge('delivery_state_tracker', 'delivery_state_db')

    return components, graph


def simulate_transaction(transaction, components, metrics):
    """
    Simulates the processing of a transaction through the involved components.
    """
    if transaction.transaction_type == 'alert_trigger':
        # alert_trigger: api_gateway → data_validator → event_bus → alert_rules_engine_ms → alerts_events_db → notification_emitter → external_service
        channel_comp = transaction.channel if transaction.channel in components else 'sms_service'
        components_sequence = ['api_gateway', 'data_validator', 'event_bus', 'alert_rules_engine_ms', 'alerts_events_db', 'notification_emitter', channel_comp]
    elif transaction.transaction_type == 'status_query':
        # status_query: api_gateway → event_bus → delivery_state_tracker → delivery_state_db
        components_sequence = ['api_gateway', 'event_bus', 'delivery_state_tracker', 'delivery_state_db']
    else:
        components_sequence = []

    transaction.components_involved = components_sequence
    success = True
    for comp_name in components_sequence:
        component = components.get(comp_name)
        if component:
            if not component.process_transaction(transaction):
                success = False
                transaction.status = 'Failed'
                break
        else:
            success = False
            transaction.status = 'Failed'
            break
    if success:
        transaction.status = 'Success'
    metrics.append(transaction)


def induce_failure(components):
    """
    Induces a failure in the architecture by reducing the capacity of a critical component.
    The event_bus is the most critical component in the subsystem (availability: critical).
    """
    critical_component = components.get('event_bus')
    if critical_component:
        critical_component.capacity = 50
        print("\n[Failure Scenario] Capacity of 'event_bus' reduced to 50.")


# Ejecutar simulación

In [24]:
index = 0
current_weight = 1
max_weight = 3
# Para ejecutar la función, simplemente llámala:
run_notification_simulation()


=== Simulating Transactions with Initial Architecture ===

=== Simulation Results ===
Total Transactions: 2000
Successful Transactions: 1247
Failed Transactions: 753

Summary by Transaction Type:
  Alert_trigger: 654 successful, 546 failed
  Status_query: 593 successful, 207 failed

Summary by Component:
  api_gateway: Processed 2000, Failed 753
  data_validator: Processed 1200, Failed 546
  event_bus: Processed 2000, Failed 753
  alert_rules_engine_ms: Processed 1200, Failed 546
  alerts_events_db: Processed 1200, Failed 546
  notification_emitter: Processed 1200, Failed 546
  sms_service: Processed 406, Failed 173
  notification_push: Processed 393, Failed 179
  cap_radio_difussion: Processed 401, Failed 194
  delivery_state_tracker: Processed 800, Failed 207
  delivery_state_db: Processed 800, Failed 207

Details of Failed Transactions:
Source: c4_system via sms_service, Type: alert_trigger, Components: ['api_gateway', 'data_validator', 'event_bus', 'alert_rules_engine_ms', 'alerts

# Balanceador de carga básico

Distribuye la carga de manera aleatoria entre los servidores disponibles. Aunque es simple, puede ser eficiente en ciertos escenarios donde no hay una diferencia significativa en la carga de los servidores.

En este caso, se replica el `notification_emitter` (2 instancias, como en el modelo .sos) y se añade un balanceador básico.

In [ ]:
def setup_architecture():
    """
    Sets up the system architecture by instantiating components and defining their connections.
    With a basic load balancer for notification_emitter.
    """
    components = {}
    graph = nx.DiGraph()

    # Presentation Tier
    components['api_gateway'] = PresentationTier('api_gateway', capacity=200)

    # Logic Tier
    components['data_validator'] = LogicTier('data_validator', capacity=200)
    components['event_bus'] = LogicTier('event_bus', capacity=500)
    components['alert_rules_engine_ms'] = LogicTier('alert_rules_engine_ms', capacity=300)

    # Load Balancer for notification_emitter
    components['notification_lb'] = LogicTier('notification_lb', capacity=10000)  # ----------------> New LoadBalancer

    components['notification_emitter_1'] = LogicTier('notification_emitter_1', capacity=300)  # ----------------> New notification_emitter_1
    components['notification_emitter_2'] = LogicTier('notification_emitter_2', capacity=500)  # ----------------> New notification_emitter_2

    components['delivery_state_tracker'] = LogicTier('delivery_state_tracker', capacity=200)

    # External Service Tier
    components['sms_service'] = ExternalServiceTier('sms_service', capacity=500)
    components['cap_radio_difussion'] = ExternalServiceTier('cap_radio_difussion', capacity=300)
    components['notification_push'] = ExternalServiceTier('notification_push', capacity=400)

    # Data Tier
    components['alerts_events_db'] = DataTier('alerts_events_db', capacity=300)
    components['delivery_state_db'] = DataTier('delivery_state_db', capacity=300)
    components['alerts_events_cache'] = DataTier('alerts_events_cache', capacity=400)

    # Define connections between components
    graph.add_edge('api_gateway', 'data_validator')
    graph.add_edge('data_validator', 'event_bus')
    graph.add_edge('event_bus', 'alert_rules_engine_ms')
    graph.add_edge('event_bus', 'notification_lb')           # ------------------> Change in flow
    graph.add_edge('event_bus', 'delivery_state_tracker')
    graph.add_edge('alert_rules_engine_ms', 'alerts_events_db')
    graph.add_edge('alert_rules_engine_ms', 'alerts_events_cache')
    graph.add_edge('notification_lb', 'notification_emitter_1')   # ------------------> Change in flow
    graph.add_edge('notification_lb', 'notification_emitter_2')   # ------------------> Change in flow
    graph.add_edge('notification_emitter_1', 'sms_service')       # ------------------> Change in flow
    graph.add_edge('notification_emitter_1', 'cap_radio_difussion') # ------------------> Change in flow
    graph.add_edge('notification_emitter_1', 'notification_push') # ------------------> Change in flow
    graph.add_edge('notification_emitter_2', 'sms_service')       # ------------------> Change in flow
    graph.add_edge('notification_emitter_2', 'cap_radio_difussion') # ------------------> Change in flow
    graph.add_edge('notification_emitter_2', 'notification_push') # ------------------> Change in flow
    graph.add_edge('notification_emitter_1', 'delivery_state_tracker') # ------------------> Change in flow
    graph.add_edge('notification_emitter_2', 'delivery_state_tracker') # ------------------> Change in flow
    graph.add_edge('delivery_state_tracker', 'delivery_state_db')

    return components, graph


def simulate_transaction(transaction, components, metrics):
    """
    Simulates the processing of a transaction through the involved components.
    """
    if transaction.transaction_type == 'alert_trigger':
        channel_comp = transaction.channel if transaction.channel in components else 'sms_service'
        emitter = random.choice(['notification_emitter_1', 'notification_emitter_2'])
        components_sequence = ['api_gateway', 'data_validator', 'event_bus', 'alert_rules_engine_ms', 'alerts_events_db', 'notification_lb', emitter, channel_comp]
    elif transaction.transaction_type == 'status_query':
        components_sequence = ['api_gateway', 'event_bus', 'delivery_state_tracker', 'delivery_state_db']
    else:
        components_sequence = []

    transaction.components_involved = components_sequence
    success = True
    for comp_name in components_sequence:
        component = components.get(comp_name)
        if component:
            if not component.process_transaction(transaction):
                success = False
                transaction.status = 'Failed'
                break
        else:
            success = False
            transaction.status = 'Failed'
            break
    if success:
        transaction.status = 'Success'
    metrics.append(transaction)


def induce_failure(components):
    """
    Induces a failure in the architecture by reducing the capacity of a critical component.
    """
    critical_component = components.get('notification_emitter_1')
    if critical_component:
        critical_component.capacity = 50
        print("\n[Failure Scenario] Capacity of 'notification_emitter_1' reduced to 50.")


# Balanceador de carga múltiple

Distribuye la carga de manera aleatoria entre los servidores disponibles en múltiples capas de la arquitectura. Se añaden balanceadores para `api_gateway`, `event_bus` y `notification_emitter`.

In [ ]:
def setup_architecture():
    """
    Sets up the system architecture by instantiating components and defining their connections.
    With multiple load balancers at different tiers.
    """
    components = {}
    graph = nx.DiGraph()

    # Presentation Tier
    components['api_gateway_lb'] = PresentationTier('api_gateway_lb', capacity=500)  # ----------------> New LoadBalancer
    components['api_gateway_1'] = PresentationTier('api_gateway_1', capacity=200)   # ----------------> New api_gateway_1
    components['api_gateway_2'] = PresentationTier('api_gateway_2', capacity=200)   # ----------------> New api_gateway_2

    # Logic Tier
    components['data_validator'] = LogicTier('data_validator', capacity=200)

    components['event_bus_lb'] = LogicTier('event_bus_lb', capacity=500)            # ----------------> New LoadBalancer
    components['event_bus_1'] = LogicTier('event_bus_1', capacity=300)              # ----------------> New event_bus_1
    components['event_bus_2'] = LogicTier('event_bus_2', capacity=300)              # ----------------> New event_bus_2
    components['event_bus_3'] = LogicTier('event_bus_3', capacity=300)              # ----------------> New event_bus_3 (3 replicas as in model)

    components['alert_rules_engine_ms_1'] = LogicTier('alert_rules_engine_ms_1', capacity=300)  # ----------------> New
    components['alert_rules_engine_ms_2'] = LogicTier('alert_rules_engine_ms_2', capacity=300)  # ----------------> New

    components['notification_lb'] = LogicTier('notification_lb', capacity=500)      # ----------------> New LoadBalancer
    components['notification_emitter_1'] = LogicTier('notification_emitter_1', capacity=300)   # ----------------> New
    components['notification_emitter_2'] = LogicTier('notification_emitter_2', capacity=300)   # ----------------> New

    components['delivery_state_tracker_1'] = LogicTier('delivery_state_tracker_1', capacity=200)  # ----------------> New
    components['delivery_state_tracker_2'] = LogicTier('delivery_state_tracker_2', capacity=200)  # ----------------> New

    # External Service Tier
    components['sms_service'] = ExternalServiceTier('sms_service', capacity=500)
    components['cap_radio_difussion'] = ExternalServiceTier('cap_radio_difussion', capacity=300)
    components['notification_push'] = ExternalServiceTier('notification_push', capacity=400)

    # Data Tier
    components['alerts_events_db'] = DataTier('alerts_events_db', capacity=300)
    components['delivery_state_db'] = DataTier('delivery_state_db', capacity=300)
    components['alerts_events_cache'] = DataTier('alerts_events_cache', capacity=400)

    # Define connections between components
    graph.add_edge('api_gateway_lb', 'api_gateway_1')           # ------------------> Change in flow
    graph.add_edge('api_gateway_lb', 'api_gateway_2')           # ------------------> Change in flow
    graph.add_edge('api_gateway_1', 'data_validator')
    graph.add_edge('api_gateway_2', 'data_validator')

    graph.add_edge('data_validator', 'event_bus_lb')            # ------------------> Change in flow
    graph.add_edge('event_bus_lb', 'event_bus_1')               # ------------------> Change in flow
    graph.add_edge('event_bus_lb', 'event_bus_2')               # ------------------> Change in flow
    graph.add_edge('event_bus_lb', 'event_bus_3')               # ------------------> Change in flow

    graph.add_edge('event_bus_1', 'alert_rules_engine_ms_1')
    graph.add_edge('event_bus_1', 'alert_rules_engine_ms_2')
    graph.add_edge('event_bus_2', 'alert_rules_engine_ms_1')
    graph.add_edge('event_bus_2', 'alert_rules_engine_ms_2')
    graph.add_edge('event_bus_3', 'alert_rules_engine_ms_1')
    graph.add_edge('event_bus_3', 'alert_rules_engine_ms_2')

    graph.add_edge('event_bus_1', 'notification_lb')            # ------------------> Change in flow
    graph.add_edge('event_bus_2', 'notification_lb')
    graph.add_edge('event_bus_3', 'notification_lb')
    graph.add_edge('event_bus_1', 'delivery_state_tracker_1')
    graph.add_edge('event_bus_1', 'delivery_state_tracker_2')
    graph.add_edge('event_bus_2', 'delivery_state_tracker_1')
    graph.add_edge('event_bus_2', 'delivery_state_tracker_2')
    graph.add_edge('event_bus_3', 'delivery_state_tracker_1')
    graph.add_edge('event_bus_3', 'delivery_state_tracker_2')

    graph.add_edge('alert_rules_engine_ms_1', 'alerts_events_db')
    graph.add_edge('alert_rules_engine_ms_1', 'alerts_events_cache')
    graph.add_edge('alert_rules_engine_ms_2', 'alerts_events_db')
    graph.add_edge('alert_rules_engine_ms_2', 'alerts_events_cache')

    graph.add_edge('notification_lb', 'notification_emitter_1') # ------------------> Change in flow
    graph.add_edge('notification_lb', 'notification_emitter_2') # ------------------> Change in flow
    graph.add_edge('notification_emitter_1', 'sms_service')
    graph.add_edge('notification_emitter_1', 'cap_radio_difussion')
    graph.add_edge('notification_emitter_1', 'notification_push')
    graph.add_edge('notification_emitter_2', 'sms_service')
    graph.add_edge('notification_emitter_2', 'cap_radio_difussion')
    graph.add_edge('notification_emitter_2', 'notification_push')
    graph.add_edge('notification_emitter_1', 'delivery_state_tracker_1')
    graph.add_edge('notification_emitter_1', 'delivery_state_tracker_2')
    graph.add_edge('notification_emitter_2', 'delivery_state_tracker_1')
    graph.add_edge('notification_emitter_2', 'delivery_state_tracker_2')

    graph.add_edge('delivery_state_tracker_1', 'delivery_state_db')
    graph.add_edge('delivery_state_tracker_2', 'delivery_state_db')

    return components, graph


def simulate_transaction(transaction, components, metrics):
    """
    Simulates the processing of a transaction through the involved components.
    """
    if transaction.transaction_type == 'alert_trigger':
        channel_comp = transaction.channel if transaction.channel in components else 'sms_service'
        components_sequence = [
            'api_gateway_lb',
            random.choice(['api_gateway_1', 'api_gateway_2']),
            'data_validator',
            'event_bus_lb',
            random.choice(['event_bus_1', 'event_bus_2', 'event_bus_3']),
            random.choice(['alert_rules_engine_ms_1', 'alert_rules_engine_ms_2']),
            'alerts_events_db',
            'notification_lb',
            random.choice(['notification_emitter_1', 'notification_emitter_2']),
            channel_comp
        ]
    elif transaction.transaction_type == 'status_query':
        components_sequence = [
            'api_gateway_lb',
            random.choice(['api_gateway_1', 'api_gateway_2']),
            'event_bus_lb',
            random.choice(['event_bus_1', 'event_bus_2', 'event_bus_3']),
            random.choice(['delivery_state_tracker_1', 'delivery_state_tracker_2']),
            'delivery_state_db'
        ]
    else:
        components_sequence = []

    transaction.components_involved = components_sequence
    success = True
    for comp_name in components_sequence:
        component = components.get(comp_name)
        if component:
            if not component.process_transaction(transaction):
                success = False
                transaction.status = 'Failed'
                break
        else:
            success = False
            transaction.status = 'Failed'
            break
    if success:
        transaction.status = 'Success'
    metrics.append(transaction)


def induce_failure(components):
    """
    Induces a failure in the architecture by reducing the capacity of a critical component.
    """
    critical_component = components.get('event_bus_1')
    if critical_component:
        critical_component.capacity = 50
        print("\n[Failure Scenario] Capacity of 'event_bus_1' reduced to 50.")


# Round Robin

Este método distribuye las solicitudes de manera uniforme entre todos los servidores disponibles en forma de ronda. Cada vez que llega una nueva solicitud, el balanceador de carga la envía al siguiente servidor en la lista. Una vez que se llega al final de la lista, se comienza de nuevo desde el primer servidor.

Aplica Round Robin sobre las 3 réplicas del `event_bus` (como indica el modelo .sos) y sobre las 2 réplicas del `notification_emitter`.

In [ ]:
def round_robin_lb(names):
    global index  # Reference to the global variable
    if not names:
        return None  # Returns None if the list is empty

    name = names[index]  # Gets the current name
    index = (index + 1) % len(names)  # Updates the index cyclically
    return name

def setup_architecture():
    """
    Sets up the system architecture by instantiating components and defining their connections.
    With Round Robin load balancing for event_bus and notification_emitter.
    """
    components = {}
    graph = nx.DiGraph()

    # Presentation Tier
    components['api_gateway'] = PresentationTier('api_gateway', capacity=200)

    # Logic Tier
    components['data_validator'] = LogicTier('data_validator', capacity=200)
    components['event_bus_lb'] = LogicTier('event_bus_lb', capacity=500)  # ----------------> New LoadBalancer
    components['event_bus_1'] = LogicTier('event_bus_1', capacity=300)   # ----------------> New event_bus_1
    components['event_bus_2'] = LogicTier('event_bus_2', capacity=300)   # ----------------> New event_bus_2
    components['event_bus_3'] = LogicTier('event_bus_3', capacity=300)   # ----------------> New event_bus_3 (3 replicas)
    components['alert_rules_engine_ms'] = LogicTier('alert_rules_engine_ms', capacity=300)

    components['notification_lb'] = LogicTier('notification_lb', capacity=500)  # ----------------> New LoadBalancer
    components['notification_emitter_1'] = LogicTier('notification_emitter_1', capacity=300)  # ----------------> New
    components['notification_emitter_2'] = LogicTier('notification_emitter_2', capacity=300)  # ----------------> New

    components['delivery_state_tracker'] = LogicTier('delivery_state_tracker', capacity=200)

    # External Service Tier
    components['sms_service'] = ExternalServiceTier('sms_service', capacity=500)
    components['cap_radio_difussion'] = ExternalServiceTier('cap_radio_difussion', capacity=300)
    components['notification_push'] = ExternalServiceTier('notification_push', capacity=400)

    # Data Tier
    components['alerts_events_db'] = DataTier('alerts_events_db', capacity=300)
    components['delivery_state_db'] = DataTier('delivery_state_db', capacity=300)
    components['alerts_events_cache'] = DataTier('alerts_events_cache', capacity=400)

    # Define connections between components
    graph.add_edge('api_gateway', 'data_validator')
    graph.add_edge('data_validator', 'event_bus_lb')          # ------------------> Change in flow
    graph.add_edge('event_bus_lb', 'event_bus_1')             # ------------------> Change in flow
    graph.add_edge('event_bus_lb', 'event_bus_2')             # ------------------> Change in flow
    graph.add_edge('event_bus_lb', 'event_bus_3')             # ------------------> Change in flow
    graph.add_edge('event_bus_1', 'alert_rules_engine_ms')
    graph.add_edge('event_bus_2', 'alert_rules_engine_ms')
    graph.add_edge('event_bus_3', 'alert_rules_engine_ms')
    graph.add_edge('event_bus_1', 'notification_lb')          # ------------------> Change in flow
    graph.add_edge('event_bus_2', 'notification_lb')
    graph.add_edge('event_bus_3', 'notification_lb')
    graph.add_edge('event_bus_1', 'delivery_state_tracker')
    graph.add_edge('event_bus_2', 'delivery_state_tracker')
    graph.add_edge('event_bus_3', 'delivery_state_tracker')
    graph.add_edge('alert_rules_engine_ms', 'alerts_events_db')
    graph.add_edge('alert_rules_engine_ms', 'alerts_events_cache')
    graph.add_edge('notification_lb', 'notification_emitter_1')  # ------------------> Change in flow
    graph.add_edge('notification_lb', 'notification_emitter_2')  # ------------------> Change in flow
    graph.add_edge('notification_emitter_1', 'sms_service')
    graph.add_edge('notification_emitter_1', 'cap_radio_difussion')
    graph.add_edge('notification_emitter_1', 'notification_push')
    graph.add_edge('notification_emitter_2', 'sms_service')
    graph.add_edge('notification_emitter_2', 'cap_radio_difussion')
    graph.add_edge('notification_emitter_2', 'notification_push')
    graph.add_edge('notification_emitter_1', 'delivery_state_tracker')
    graph.add_edge('notification_emitter_2', 'delivery_state_tracker')
    graph.add_edge('delivery_state_tracker', 'delivery_state_db')

    return components, graph


def simulate_transaction(transaction, components, metrics):
    """
    Simulates the processing of a transaction through the involved components.
    """
    if transaction.transaction_type == 'alert_trigger':
        channel_comp = transaction.channel if transaction.channel in components else 'sms_service'
        components_sequence = [
            'api_gateway',
            'data_validator',
            'event_bus_lb',
            round_robin_lb(['event_bus_1', 'event_bus_2', 'event_bus_3']),
            'alert_rules_engine_ms',
            'alerts_events_db',
            'notification_lb',
            round_robin_lb(['notification_emitter_1', 'notification_emitter_2']),
            channel_comp
        ]
    elif transaction.transaction_type == 'status_query':
        components_sequence = [
            'api_gateway',
            'event_bus_lb',
            round_robin_lb(['event_bus_1', 'event_bus_2', 'event_bus_3']),
            'delivery_state_tracker',
            'delivery_state_db'
        ]
    else:
        components_sequence = []

    transaction.components_involved = components_sequence
    success = True
    for comp_name in components_sequence:
        component = components.get(comp_name)
        if component:
            if not component.process_transaction(transaction):
                success = False
                transaction.status = 'Failed'
                break
        else:
            success = False
            transaction.status = 'Failed'
            break
    if success:
        transaction.status = 'Success'
    metrics.append(transaction)


def induce_failure(components):
    """
    Induces a failure in the architecture by reducing the capacity of a critical component.
    """
    critical_component = components.get('event_bus_1')
    if critical_component:
        critical_component.capacity = 50
        print("\n[Failure Scenario] Capacity of 'event_bus_1' reduced to 50.")


# Weighted Round Robin

Similar al Round Robin, pero asigna "pesos" a los servidores. Aunque todos los servidores reciben solicitudes, aquellos con un peso mayor recibirán más solicitudes que los que tienen un peso menor. Esto permite manejar servidores con diferentes capacidades de rendimiento.

En este caso, las réplicas del `event_bus` (3 instancias) tendrán pesos diferentes, y las del `notification_emitter` (2 instancias) también.

In [ ]:
def weighted_round_robin_lb(names, weights):
    global index, current_weight, max_weight

    # Check if the lists are empty
    if not names or not weights or len(names) != len(weights):
        return None

    # Set the maximum weight if not already done
    if max_weight == 0:
        max_weight = max(weights)

    while True:
        # Updates the index cyclically
        index = (index + 1) % len(names)

        if index == 0:
            # Increase current weight after completing a lap
            current_weight = (current_weight + 1) % (max_weight + 1)

        # If the current weight 0 is reached, continue to the next
        if current_weight == 0:
            continue

        # Check if the weight of the current name is greater than or equal to the current weight
        if weights[index] >= current_weight:
            return names[index]

def setup_architecture():
    """
    Sets up the system architecture by instantiating components and defining their connections.
    With Weighted Round Robin load balancing.
    """
    components = {}
    graph = nx.DiGraph()

    # Presentation Tier
    components['api_gateway'] = PresentationTier('api_gateway', capacity=200)

    # Logic Tier
    components['data_validator'] = LogicTier('data_validator', capacity=200)
    components['event_bus_lb'] = LogicTier('event_bus_lb', capacity=500)  # ----------------> New LoadBalancer
    components['event_bus_1'] = LogicTier('event_bus_1', capacity=300)   # ----------------> New event_bus_1
    components['event_bus_2'] = LogicTier('event_bus_2', capacity=300)   # ----------------> New event_bus_2
    components['event_bus_3'] = LogicTier('event_bus_3', capacity=300)   # ----------------> New event_bus_3 (3 replicas)
    components['alert_rules_engine_ms'] = LogicTier('alert_rules_engine_ms', capacity=300)

    components['notification_lb'] = LogicTier('notification_lb', capacity=500)  # ----------------> New LoadBalancer
    components['notification_emitter_1'] = LogicTier('notification_emitter_1', capacity=300)  # ----------------> New
    components['notification_emitter_2'] = LogicTier('notification_emitter_2', capacity=300)  # ----------------> New

    components['delivery_state_tracker'] = LogicTier('delivery_state_tracker', capacity=200)

    # External Service Tier
    components['sms_service'] = ExternalServiceTier('sms_service', capacity=500)
    components['cap_radio_difussion'] = ExternalServiceTier('cap_radio_difussion', capacity=300)
    components['notification_push'] = ExternalServiceTier('notification_push', capacity=400)

    # Data Tier
    components['alerts_events_db'] = DataTier('alerts_events_db', capacity=300)
    components['delivery_state_db'] = DataTier('delivery_state_db', capacity=300)
    components['alerts_events_cache'] = DataTier('alerts_events_cache', capacity=400)

    # Define connections between components
    graph.add_edge('api_gateway', 'data_validator')
    graph.add_edge('data_validator', 'event_bus_lb')          # ------------------> Change in flow
    graph.add_edge('event_bus_lb', 'event_bus_1')             # ------------------> Change in flow
    graph.add_edge('event_bus_lb', 'event_bus_2')             # ------------------> Change in flow
    graph.add_edge('event_bus_lb', 'event_bus_3')             # ------------------> Change in flow
    graph.add_edge('event_bus_1', 'alert_rules_engine_ms')
    graph.add_edge('event_bus_2', 'alert_rules_engine_ms')
    graph.add_edge('event_bus_3', 'alert_rules_engine_ms')
    graph.add_edge('event_bus_1', 'notification_lb')          # ------------------> Change in flow
    graph.add_edge('event_bus_2', 'notification_lb')
    graph.add_edge('event_bus_3', 'notification_lb')
    graph.add_edge('event_bus_1', 'delivery_state_tracker')
    graph.add_edge('event_bus_2', 'delivery_state_tracker')
    graph.add_edge('event_bus_3', 'delivery_state_tracker')
    graph.add_edge('alert_rules_engine_ms', 'alerts_events_db')
    graph.add_edge('alert_rules_engine_ms', 'alerts_events_cache')
    graph.add_edge('notification_lb', 'notification_emitter_1')  # ------------------> Change in flow
    graph.add_edge('notification_lb', 'notification_emitter_2')  # ------------------> Change in flow
    graph.add_edge('notification_emitter_1', 'sms_service')
    graph.add_edge('notification_emitter_1', 'cap_radio_difussion')
    graph.add_edge('notification_emitter_1', 'notification_push')
    graph.add_edge('notification_emitter_2', 'sms_service')
    graph.add_edge('notification_emitter_2', 'cap_radio_difussion')
    graph.add_edge('notification_emitter_2', 'notification_push')
    graph.add_edge('notification_emitter_1', 'delivery_state_tracker')
    graph.add_edge('notification_emitter_2', 'delivery_state_tracker')
    graph.add_edge('delivery_state_tracker', 'delivery_state_db')

    return components, graph


def simulate_transaction(transaction, components, metrics):
    """
    Simulates the processing of a transaction through the involved components.
    """
    # Weights for event_bus replicas (3 replicas, first has lower weight, others higher)
    event_bus_names = ['event_bus_1', 'event_bus_2', 'event_bus_3']
    event_bus_weights = [1, 2, 3]  # event_bus_3 receives more load (highest weight)

    # Weights for notification_emitter replicas (2 replicas)
    emitter_names = ['notification_emitter_1', 'notification_emitter_2']
    emitter_weights = [1, 2]  # notification_emitter_2 receives more load

    if transaction.transaction_type == 'alert_trigger':
        channel_comp = transaction.channel if transaction.channel in components else 'sms_service'
        components_sequence = [
            'api_gateway',
            'data_validator',
            'event_bus_lb',
            weighted_round_robin_lb(event_bus_names, event_bus_weights),
            'alert_rules_engine_ms',
            'alerts_events_db',
            'notification_lb',
            weighted_round_robin_lb(emitter_names, emitter_weights),
            channel_comp
        ]
    elif transaction.transaction_type == 'status_query':
        components_sequence = [
            'api_gateway',
            'event_bus_lb',
            weighted_round_robin_lb(event_bus_names, event_bus_weights),
            'delivery_state_tracker',
            'delivery_state_db'
        ]
    else:
        components_sequence = []

    transaction.components_involved = components_sequence
    success = True
    for comp_name in components_sequence:
        component = components.get(comp_name)
        if component:
            if not component.process_transaction(transaction):
                success = False
                transaction.status = 'Failed'
                break
        else:
            success = False
            transaction.status = 'Failed'
            break
    if success:
        transaction.status = 'Success'
    metrics.append(transaction)


def induce_failure(components):
    """
    Induces a failure in the architecture by reducing the capacity of a critical component.
    """
    critical_component = components.get('event_bus_1')
    if critical_component:
        critical_component.capacity = 50
        print("\n[Failure Scenario] Capacity of 'event_bus_1' reduced to 50.")


# Escenario de Calidad: Fallo de Bases de Datos en Nube

El proveedor de infraestructura cloud alerta que los servicios de bases de datos están funcionando de manera intermitente.

| Atributo | Descripción |
|---|---|
| **Estímulo** | Los servicios de bases de datos fallan aleatoriamente en el 50% de las solicitudes |
| **Fuente** | Proveedor de infraestructura cloud |
| **Entorno** | Operación normal bajo carga estándar (2000 transacciones concurrentes) |
| **Artefacto** | `alerts_events_db`, `delivery_state_db`, `alerts_events_cache` |
| **Respuesta** | El sistema procesa o rechaza transacciones de forma determinista, sin corrupción de datos |
| **Medida** | Tasa de éxito de transacciones con P(fallo_BD) = 0.5 |

In [25]:
# ================================
# Escenario de Calidad — Fallo de Bases de Datos
# ================================

DB_FAILURE_RATE = 0.5  # 50% de probabilidad de fallo por acceso a BD

class UnreliableComponent(Component):
    """Component that fails randomly at a given rate (simulates flaky cloud DB services)."""
    def __init__(self, name, component_type, capacity=100, failure_rate=0.0):
        super().__init__(name, component_type, capacity)
        self.failure_rate = failure_rate

    def process_transaction(self, transaction):
        if random.random() < self.failure_rate:
            return False  # Fallo aleatorio independiente de la carga
        return super().process_transaction(transaction)


def setup_architecture_db_failure():
    """Arquitectura original con Data Tier no confiable (50% failure rate)."""
    components = {}
    graph = nx.DiGraph()

    components['api_gateway'] = PresentationTier('api_gateway', capacity=200)
    components['data_validator'] = LogicTier('data_validator', capacity=200)
    components['event_bus'] = LogicTier('event_bus', capacity=500)
    components['alert_rules_engine_ms'] = LogicTier('alert_rules_engine_ms', capacity=300)
    components['notification_emitter'] = LogicTier('notification_emitter', capacity=300)
    components['delivery_state_tracker'] = LogicTier('delivery_state_tracker', capacity=200)

    components['sms_service'] = ExternalServiceTier('sms_service', capacity=500)
    components['cap_radio_difussion'] = ExternalServiceTier('cap_radio_difussion', capacity=300)
    components['notification_push'] = ExternalServiceTier('notification_push', capacity=400)

    # Data Tier con failure_rate=0.5 — simula inestabilidad del proveedor cloud
    components['alerts_events_db'] = UnreliableComponent('alerts_events_db', 'Data Tier', capacity=300, failure_rate=DB_FAILURE_RATE)
    components['delivery_state_db'] = UnreliableComponent('delivery_state_db', 'Data Tier', capacity=300, failure_rate=DB_FAILURE_RATE)
    components['alerts_events_cache'] = UnreliableComponent('alerts_events_cache', 'Data Tier', capacity=400, failure_rate=DB_FAILURE_RATE)

    graph.add_edge('api_gateway', 'data_validator')
    graph.add_edge('data_validator', 'event_bus')
    graph.add_edge('event_bus', 'alert_rules_engine_ms')
    graph.add_edge('event_bus', 'notification_emitter')
    graph.add_edge('event_bus', 'delivery_state_tracker')
    graph.add_edge('alert_rules_engine_ms', 'alerts_events_db')
    graph.add_edge('alert_rules_engine_ms', 'alerts_events_cache')
    graph.add_edge('notification_emitter', 'sms_service')
    graph.add_edge('notification_emitter', 'cap_radio_difussion')
    graph.add_edge('notification_emitter', 'notification_push')
    graph.add_edge('notification_emitter', 'delivery_state_tracker')
    graph.add_edge('delivery_state_tracker', 'delivery_state_db')

    return components, graph


def simulate_transaction_db_failure(transaction, components, metrics):
    """Mismos flujos que la arquitectura original."""
    if transaction.transaction_type == 'alert_trigger':
        channel_comp = transaction.channel if transaction.channel in components else 'sms_service'
        components_sequence = [
            'api_gateway', 'data_validator', 'event_bus',
            'alert_rules_engine_ms', 'alerts_events_db',
            'notification_emitter', channel_comp
        ]
    elif transaction.transaction_type == 'status_query':
        components_sequence = [
            'api_gateway', 'event_bus',
            'delivery_state_tracker', 'delivery_state_db'
        ]
    else:
        components_sequence = []

    transaction.components_involved = components_sequence
    success = True
    for comp_name in components_sequence:
        component = components.get(comp_name)
        if component:
            if not component.process_transaction(transaction):
                success = False
                transaction.status = 'Failed'
                break
        else:
            success = False
            transaction.status = 'Failed'
            break
    if success:
        transaction.status = 'Success'
    metrics.append(transaction)


def run_db_failure_scenario():
    print(f"=== Escenario de Calidad: Fallo de Bases de Datos (P={DB_FAILURE_RATE*100:.0f}%) ===")

    components, graph = setup_architecture_db_failure()
    metrics = []
    channels = ['sms_service', 'cap_radio_difussion', 'notification_push']
    threads = []

    for i in range(1, 1201):
        source = 'data_adquisition_system' if i % 2 == 0 else 'c4_system'
        tx = Transaction(source, 'alert_trigger', random.choice(channels))
        t = threading.Thread(target=simulate_transaction_db_failure, args=(tx, components, metrics))
        threads.append(t)
        t.start()

    for i in range(1, 801):
        tx = Transaction('personnel_orchestration_system', 'status_query')
        t = threading.Thread(target=simulate_transaction_db_failure, args=(tx, components, metrics))
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    report_metrics(metrics)


run_db_failure_scenario()

=== Escenario de Calidad: Fallo de Bases de Datos (P=50%) ===

=== Simulation Results ===
Total Transactions: 2000
Successful Transactions: 698
Failed Transactions: 1302

Summary by Transaction Type:
  Alert_trigger: 379 successful, 821 failed
  Status_query: 319 successful, 481 failed

Summary by Component:
  api_gateway: Processed 2000, Failed 1302
  data_validator: Processed 1200, Failed 821
  event_bus: Processed 2000, Failed 1302
  alert_rules_engine_ms: Processed 1200, Failed 821
  alerts_events_db: Processed 1200, Failed 821
  notification_emitter: Processed 1200, Failed 821
  cap_radio_difussion: Processed 410, Failed 277
  notification_push: Processed 404, Failed 287
  sms_service: Processed 386, Failed 257
  delivery_state_tracker: Processed 800, Failed 481
  delivery_state_db: Processed 800, Failed 481

Details of Failed Transactions:
Source: c4_system via cap_radio_difussion, Type: alert_trigger, Components: ['api_gateway', 'data_validator', 'event_bus', 'alert_rules_engine

# Táctica Arquitectónica: Resiliencia ante Fallo de BD

Dado el escenario anterior (50% de fallo en bases de datos), se aplican las siguientes tácticas:

| Táctica | Descripción |
|---|---|
| **Multi-cloud failover** | Réplica de cada BD en un proveedor secundario independiente. Si el primario falla, se redirige el tráfico automáticamente (DNS failover inteligente) |
| **Cache fallback** | Si `alerts_events_db` falla en ambos proveedores, `alerts_events_cache` sirve las reglas de prioridad |
| **Flujo desacoplado** | El envío de alertas al `notification_emitter` no está bloqueado por la escritura en BD — la alerta se entrega aunque se pierda el registro |

> **Hipótesis de mejora**: con fallos independientes entre proveedores (P=0.5 cada uno), la probabilidad de que ambos fallen simultáneamente es P=0.25. El flujo desacoplado elimina además las fallas de `alert_trigger` por indisponibilidad de BD.

In [26]:
# ================================
# Táctica: Resiliencia ante Fallo de BD
# ================================

def setup_architecture_resilient():
    """
    Arquitectura con tácticas de resiliencia:
    - Multi-cloud failover: primary + secondary DB por proveedor independiente
    - Cache fallback para alerts_events_db
    - Flujo desacoplado: la escritura en BD no bloquea el envío de alertas
    """
    components = {}
    graph = nx.DiGraph()

    components['api_gateway'] = PresentationTier('api_gateway', capacity=200)
    components['data_validator'] = LogicTier('data_validator', capacity=200)
    components['event_bus'] = LogicTier('event_bus', capacity=500)
    components['alert_rules_engine_ms'] = LogicTier('alert_rules_engine_ms', capacity=300)
    components['notification_emitter'] = LogicTier('notification_emitter', capacity=300)
    components['delivery_state_tracker'] = LogicTier('delivery_state_tracker', capacity=200)

    components['sms_service'] = ExternalServiceTier('sms_service', capacity=500)
    components['cap_radio_difussion'] = ExternalServiceTier('cap_radio_difussion', capacity=300)
    components['notification_push'] = ExternalServiceTier('notification_push', capacity=400)

    # Data Tier — proveedor primario (P_fallo=0.5)
    components['alerts_events_db_primary'] = UnreliableComponent(
        'alerts_events_db_primary', 'Data Tier', capacity=300, failure_rate=DB_FAILURE_RATE)
    components['delivery_state_db_primary'] = UnreliableComponent(
        'delivery_state_db_primary', 'Data Tier', capacity=300, failure_rate=DB_FAILURE_RATE)

    # Data Tier — proveedor secundario independiente (P_fallo=0.5, independiente del primario)
    components['alerts_events_db_secondary'] = UnreliableComponent(
        'alerts_events_db_secondary', 'Data Tier', capacity=300, failure_rate=DB_FAILURE_RATE)
    components['delivery_state_db_secondary'] = UnreliableComponent(
        'delivery_state_db_secondary', 'Data Tier', capacity=300, failure_rate=DB_FAILURE_RATE)

    # Cache — último recurso para reglas de prioridad (P_fallo=0.5)
    components['alerts_events_cache'] = UnreliableComponent(
        'alerts_events_cache', 'Data Tier', capacity=400, failure_rate=DB_FAILURE_RATE)

    graph.add_edge('api_gateway', 'data_validator')
    graph.add_edge('data_validator', 'event_bus')
    graph.add_edge('event_bus', 'alert_rules_engine_ms')
    graph.add_edge('event_bus', 'notification_emitter')
    graph.add_edge('event_bus', 'delivery_state_tracker')
    graph.add_edge('alert_rules_engine_ms', 'alerts_events_db_primary')
    graph.add_edge('alert_rules_engine_ms', 'alerts_events_db_secondary')
    graph.add_edge('alert_rules_engine_ms', 'alerts_events_cache')
    graph.add_edge('notification_emitter', 'sms_service')
    graph.add_edge('notification_emitter', 'cap_radio_difussion')
    graph.add_edge('notification_emitter', 'notification_push')
    graph.add_edge('notification_emitter', 'delivery_state_tracker')
    graph.add_edge('delivery_state_tracker', 'delivery_state_db_primary')
    graph.add_edge('delivery_state_tracker', 'delivery_state_db_secondary')

    return components, graph


def try_with_fallbacks(component_names, components, transaction):
    """Intenta componentes en orden; retorna True en el primero que responde.
    Modela retry automático y failover entre proveedores."""
    for name in component_names:
        comp = components.get(name)
        if comp and comp.process_transaction(transaction):
            return True
    return False


def simulate_transaction_resilient(transaction, components, metrics):
    """
    Tácticas aplicadas:
    1. alert_trigger: la escritura en BD es fire-and-forget — la alerta se envía aunque falle el registro.
       Orden de intentos para la escritura: primary → secondary → cache.
    2. status_query: failover entre proveedores primary → secondary.
    """
    success = True

    if transaction.transaction_type == 'alert_trigger':
        channel_comp = transaction.channel if transaction.channel in components else 'sms_service'
        mandatory = ['api_gateway', 'data_validator', 'event_bus', 'alert_rules_engine_ms']
        transaction.components_involved = (
            mandatory +
            ['alerts_events_db_primary|secondary|cache (fire-and-forget)'] +
            ['notification_emitter', channel_comp]
        )

        for comp_name in mandatory:
            comp = components.get(comp_name)
            if not comp or not comp.process_transaction(transaction):
                success = False
                break

        if success:
            # Táctica 1 + 2: intento de escritura en BD, no bloquea el flujo
            try_with_fallbacks(
                ['alerts_events_db_primary', 'alerts_events_db_secondary', 'alerts_events_cache'],
                components, transaction
            )
            # Táctica 3: la alerta se envía independientemente del resultado de la BD
            for comp_name in ['notification_emitter', channel_comp]:
                comp = components.get(comp_name)
                if not comp or not comp.process_transaction(transaction):
                    success = False
                    break

    elif transaction.transaction_type == 'status_query':
        mandatory = ['api_gateway', 'event_bus', 'delivery_state_tracker']
        transaction.components_involved = mandatory + ['delivery_state_db_primary|secondary']

        for comp_name in mandatory:
            comp = components.get(comp_name)
            if not comp or not comp.process_transaction(transaction):
                success = False
                break

        if success:
            # Failover entre proveedores para la consulta de estado
            if not try_with_fallbacks(
                ['delivery_state_db_primary', 'delivery_state_db_secondary'],
                components, transaction
            ):
                success = False

    transaction.status = 'Success' if success else 'Failed'
    metrics.append(transaction)


def run_resilient_scenario():
    print(f"=== Táctica: Resiliencia ante Fallo de BD (P_fallo_por_proveedor={DB_FAILURE_RATE*100:.0f}%) ===")
    print("Tácticas activas:")
    print("  1. Multi-cloud failover — primary → secondary DB (DNS failover inteligente)")
    print("  2. Cache fallback — alerts_events_cache como respaldo de alerts_events_db")
    print("  3. Flujo desacoplado — alertas se entregan aunque falle el registro en BD\n")

    components, graph = setup_architecture_resilient()
    metrics = []
    channels = ['sms_service', 'cap_radio_difussion', 'notification_push']
    threads = []

    for i in range(1, 1201):
        source = 'data_adquisition_system' if i % 2 == 0 else 'c4_system'
        tx = Transaction(source, 'alert_trigger', random.choice(channels))
        t = threading.Thread(target=simulate_transaction_resilient, args=(tx, components, metrics))
        threads.append(t)
        t.start()

    for i in range(1, 801):
        tx = Transaction('personnel_orchestration_system', 'status_query')
        t = threading.Thread(target=simulate_transaction_resilient, args=(tx, components, metrics))
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    report_metrics(metrics)


run_resilient_scenario()

=== Táctica: Resiliencia ante Fallo de BD (P_fallo_por_proveedor=50%) ===
Tácticas activas:
  1. Multi-cloud failover — primary → secondary DB (DNS failover inteligente)
  2. Cache fallback — alerts_events_cache como respaldo de alerts_events_db
  3. Flujo desacoplado — alertas se entregan aunque falle el registro en BD


=== Simulation Results ===
Total Transactions: 2000
Successful Transactions: 1099
Failed Transactions: 901

Summary by Transaction Type:
  Alert_trigger: 656 successful, 544 failed
  Status_query: 443 successful, 357 failed

Summary by Component:
  api_gateway: Processed 2000, Failed 901
  data_validator: Processed 1200, Failed 544
  event_bus: Processed 2000, Failed 901
  alert_rules_engine_ms: Processed 1200, Failed 544
  alerts_events_db_primary|secondary|cache (fire-and-forget): Processed 1200, Failed 544
  notification_emitter: Processed 1200, Failed 544
  cap_radio_difussion: Processed 392, Failed 175
  notification_push: Processed 374, Failed 182
  sms_service:

# Escenario de Calidad: Datos Atípicos desde data_adquisition_system

El sistema 1 envía datos corruptos, inconsistentes o atípicos hacia el subsistema de notificación.
El `data_validator` actúa como barrera de calidad antes de que los datos lleguen al bus de eventos.

| Atributo | Descripción |
|---|---|
| **Estímulo** | `data_adquisition_system` envía datos corruptos, inconsistentes o atípicos |
| **Fuente** | Sistema 1 (`data_adquisition_system`) |
| **Entorno** | Operación normal; 30% de los datos entrantes son atípicos |
| **Artefacto** | `data_validator` — microservicio de validación |
| **Respuesta** | El validador detecta y rechaza datos inválidos con detalle del error; Sistema 1 reintenta con datos corregidos |
| **Medida** | Tasa de datos atípicos detectados, transacciones recuperadas tras reintento, fallos definitivos |

**Atributos de calidad involucrados**: Fiabilidad · Seguridad · Consistencia · Integridad · Interoperabilidad

In [28]:
# ================================
# Escenario de Calidad — Datos Atípicos (Arquitectura Original)
# ================================

ATYPICAL_DATA_RATE        = 0.30  # 30% de alertas desde data_adquisition_system son atípicas
ATYPICAL_FAILURE_AT_RULES = 0.80  # 80% de datos atípicos causan fallo en alert_rules_engine_ms


class UnreliableRulesEngine(Component):
    """
    alert_rules_engine_ms sin validación previa:
    los datos atípicos que llegan generan fallos con alta probabilidad
    al intentar aplicar reglas sobre datos malformados.
    """
    def __init__(self, name, capacity=300, atypical_failure_rate=0.8):
        super().__init__(name, 'Logic Tier', capacity)
        self.atypical_failure_rate = atypical_failure_rate

    def process_transaction(self, tx):
        with self.lock:
            if self.current_load >= self.capacity:
                return False
            self.current_load += 1
        time.sleep(random.uniform(0.01, 0.05))
        with self.lock:
            self.current_load -= 1
        if getattr(tx, 'is_atypical', False) and random.random() < self.atypical_failure_rate:
            tx.rules_error = 'FAILURE: datos atípicos no procesables por el motor de reglas'
            return False
        return True


def setup_architecture_atypical_original():
    """Arquitectura original — data_validator sin detección especial de datos atípicos."""
    components = {}
    graph = nx.DiGraph()

    components['api_gateway']            = PresentationTier('api_gateway', capacity=200)
    components['data_validator']          = LogicTier('data_validator', capacity=200)  # Sin detección
    components['event_bus']              = LogicTier('event_bus', capacity=500)
    components['alert_rules_engine_ms']  = UnreliableRulesEngine('alert_rules_engine_ms', capacity=300,
                                               atypical_failure_rate=ATYPICAL_FAILURE_AT_RULES)
    components['notification_emitter']   = LogicTier('notification_emitter', capacity=300)
    components['delivery_state_tracker'] = LogicTier('delivery_state_tracker', capacity=200)
    components['sms_service']            = ExternalServiceTier('sms_service', capacity=500)
    components['cap_radio_difussion']    = ExternalServiceTier('cap_radio_difussion', capacity=300)
    components['notification_push']      = ExternalServiceTier('notification_push', capacity=400)
    components['alerts_events_db']       = DataTier('alerts_events_db', capacity=300)
    components['delivery_state_db']      = DataTier('delivery_state_db', capacity=300)
    components['alerts_events_cache']    = DataTier('alerts_events_cache', capacity=400)

    graph.add_edge('api_gateway',           'data_validator')
    graph.add_edge('data_validator',         'event_bus')
    graph.add_edge('event_bus',              'alert_rules_engine_ms')
    graph.add_edge('event_bus',              'notification_emitter')
    graph.add_edge('event_bus',              'delivery_state_tracker')
    graph.add_edge('alert_rules_engine_ms',  'alerts_events_db')
    graph.add_edge('alert_rules_engine_ms',  'alerts_events_cache')
    graph.add_edge('notification_emitter',   'sms_service')
    graph.add_edge('notification_emitter',   'cap_radio_difussion')
    graph.add_edge('notification_emitter',   'notification_push')
    graph.add_edge('notification_emitter',   'delivery_state_tracker')
    graph.add_edge('delivery_state_tracker', 'delivery_state_db')

    return components, graph


def simulate_atypical_original(transaction, components, metrics):
    """Flujo original: datos atípicos no se detectan en data_validator y fallan en alert_rules_engine_ms."""
    if transaction.transaction_type == 'alert_trigger':
        channel_comp = transaction.channel if transaction.channel in components else 'sms_service'
        components_sequence = [
            'api_gateway', 'data_validator', 'event_bus',
            'alert_rules_engine_ms', 'alerts_events_db',
            'notification_emitter', channel_comp
        ]
    elif transaction.transaction_type == 'status_query':
        components_sequence = ['api_gateway', 'event_bus', 'delivery_state_tracker', 'delivery_state_db']
    else:
        components_sequence = []

    transaction.components_involved = components_sequence
    success = True
    for comp_name in components_sequence:
        comp = components.get(comp_name)
        if not comp or not comp.process_transaction(transaction):
            success = False
            break
    transaction.status = 'Success' if success else 'Failed'
    metrics.append(transaction)


def run_atypical_original_scenario():
    print("=== Escenario de Calidad: Datos Atípicos — Arquitectura Original ===")
    print(f"  Tasa de datos atípicos desde data_adquisition_system : {ATYPICAL_DATA_RATE*100:.0f}%")
    print(f"  Prob. de fallo en alert_rules_engine_ms              : {ATYPICAL_FAILURE_AT_RULES*100:.0f}%")
    print("  data_validator NO detecta datos atípicos — llegan al motor de reglas sin filtrar\n")

    components, graph = setup_architecture_atypical_original()
    metrics  = []
    channels = ['sms_service', 'cap_radio_difussion', 'notification_push']
    threads  = []

    for i in range(1, 1201):
        source = 'data_adquisition_system' if i % 2 == 0 else 'c4_system'
        tx = Transaction(source, 'alert_trigger', random.choice(channels))
        tx.is_atypical = (source == 'data_adquisition_system') and (random.random() < ATYPICAL_DATA_RATE)
        t = threading.Thread(target=simulate_atypical_original, args=(tx, components, metrics))
        threads.append(t)
        t.start()

    for i in range(1, 801):
        tx = Transaction('personnel_orchestration_system', 'status_query')
        tx.is_atypical = False
        t = threading.Thread(target=simulate_atypical_original, args=(tx, components, metrics))
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    total    = len(metrics)
    success  = sum(1 for tx in metrics if tx.status == 'Success')
    failed   = sum(1 for tx in metrics if tx.status == 'Failed')
    atypical_failed  = sum(1 for tx in metrics if tx.status == 'Failed'  and getattr(tx, 'is_atypical', False))
    atypical_slipped = sum(1 for tx in metrics if tx.status == 'Success' and getattr(tx, 'is_atypical', False))

    print(f"=== Resultados ===")
    print(f"Total Transacciones : {total}")
    print(f"Exitosas            : {success}")
    print(f"Fallidas            : {failed}")
    print(f"\n--- Impacto de Datos Atípicos sin Validación ---")
    print(f"Atípicas que fallaron en alert_rules_engine_ms : {atypical_failed}  ← fallo profundo en el pipeline")
    print(f"Atípicas que pasaron sin detección             : {atypical_slipped}  ← riesgo: notificaciones incorrectas")


run_atypical_original_scenario()

=== Escenario de Calidad: Datos Atípicos — Arquitectura Original ===
  Tasa de datos atípicos desde data_adquisition_system : 30%
  Prob. de fallo en alert_rules_engine_ms              : 80%
  data_validator NO detecta datos atípicos — llegan al motor de reglas sin filtrar

=== Resultados ===
Total Transacciones : 2000
Exitosas            : 1342
Fallidas            : 658

--- Impacto de Datos Atípicos sin Validación ---
Atípicas que fallaron en alert_rules_engine_ms : 166  ← fallo profundo en el pipeline
Atípicas que pasaron sin detección             : 18  ← riesgo: notificaciones incorrectas


# Táctica Arquitectónica: Resiliencia ante Datos Atípicos

Dado el escenario anterior (30% de datos atípicos desde `data_adquisition_system`), se aplican las siguientes tácticas:

| Táctica | Descripción |
|---|---|
| **Circuit Breaker** | Si una fuente supera el umbral de rechazos consecutivos, el circuito se abre temporalmente: los datos de esa fuente se rechazan de inmediato sin procesar, protegiendo al sistema de sobrecarga |
| **Dead Letter Queue (DLQ)** | Transacciones que agotan todos los reintentos se envían a una cola de cuarentena para auditoría y reprocesamiento manual, evitando pérdida silenciosa de datos |
| **Retry con datos corregidos** | System 1 es notificado del rechazo con detalle del error; reintenta con datos limpios (ya modelado en el escenario base) |

> **Hipótesis de mejora**: el Circuit Breaker reduce el tiempo de procesamiento innecesario una vez detectado un patrón de datos corruptos. La DLQ garantiza trazabilidad total — ningún dato se pierde silenciosamente.

In [29]:
# ================================
# Táctica: Validación Reforzada + Circuit Breaker + Dead Letter Queue
# ================================

MAX_RETRIES               = 3
CIRCUIT_BREAKER_THRESHOLD = 5
CIRCUIT_BREAKER_TIMEOUT   = 2.0


class ValidationComponent(Component):
    """
    data_validator reforzado: detecta y rechaza datos atípicos de forma temprana,
    antes de que lleguen al bus de eventos. Retorna error detallado al emisor.
    """
    def __init__(self, name, capacity=100):
        super().__init__(name, 'Logic Tier', capacity)

    def process_transaction(self, tx):
        with self.lock:
            if self.current_load >= self.capacity:
                return False
            self.current_load += 1
        time.sleep(random.uniform(0.01, 0.05))
        with self.lock:
            self.current_load -= 1
        if getattr(tx, 'is_atypical', False):
            tx.validation_error = 'REJECTED: dato atípico/corrupto — fuente notificada para reintento'
            return False
        return True


class CircuitBreakerValidationComponent(ValidationComponent):
    """
    ValidationComponent con Circuit Breaker por fuente.
    Tras CIRCUIT_BREAKER_THRESHOLD rechazos consecutivos abre el circuito,
    bloqueando esa fuente sin procesar hasta que expire el timeout.
    """
    def __init__(self, name, capacity=100):
        super().__init__(name, capacity)
        self._failure_counts     = defaultdict(int)
        self._circuit_open_until = defaultdict(float)
        self._cb_lock            = threading.Lock()

    def process_transaction(self, tx):
        source = getattr(tx, 'source', 'unknown')
        with self._cb_lock:
            if time.time() < self._circuit_open_until[source]:
                tx.validation_error = f'CIRCUIT OPEN: {source} bloqueado por datos atípicos reiterados'
                tx.circuit_open = True
                return False

        result = super().process_transaction(tx)

        with self._cb_lock:
            if not result and hasattr(tx, 'validation_error'):
                self._failure_counts[source] += 1
                if self._failure_counts[source] >= CIRCUIT_BREAKER_THRESHOLD:
                    self._circuit_open_until[source] = time.time() + CIRCUIT_BREAKER_TIMEOUT
                    self._failure_counts[source] = 0
            elif result:
                self._failure_counts[source] = 0
        return result


def setup_architecture_resilient_atypical():
    """
    Arquitectura modificada:
    - data_validator → CircuitBreakerValidationComponent (detección temprana)
    - alert_rules_engine_ms → LogicTier estándar (ya no recibe datos atípicos)
    """
    components = {}
    graph = nx.DiGraph()

    components['api_gateway']            = PresentationTier('api_gateway', capacity=200)
    components['data_validator']          = CircuitBreakerValidationComponent('data_validator', capacity=200)
    components['event_bus']              = LogicTier('event_bus', capacity=500)
    components['alert_rules_engine_ms']  = LogicTier('alert_rules_engine_ms', capacity=300)
    components['notification_emitter']   = LogicTier('notification_emitter', capacity=300)
    components['delivery_state_tracker'] = LogicTier('delivery_state_tracker', capacity=200)
    components['sms_service']            = ExternalServiceTier('sms_service', capacity=500)
    components['cap_radio_difussion']    = ExternalServiceTier('cap_radio_difussion', capacity=300)
    components['notification_push']      = ExternalServiceTier('notification_push', capacity=400)
    components['alerts_events_db']       = DataTier('alerts_events_db', capacity=300)
    components['delivery_state_db']      = DataTier('delivery_state_db', capacity=300)
    components['alerts_events_cache']    = DataTier('alerts_events_cache', capacity=400)

    graph.add_edge('api_gateway',           'data_validator')
    graph.add_edge('data_validator',         'event_bus')
    graph.add_edge('event_bus',              'alert_rules_engine_ms')
    graph.add_edge('event_bus',              'notification_emitter')
    graph.add_edge('event_bus',              'delivery_state_tracker')
    graph.add_edge('alert_rules_engine_ms',  'alerts_events_db')
    graph.add_edge('alert_rules_engine_ms',  'alerts_events_cache')
    graph.add_edge('notification_emitter',   'sms_service')
    graph.add_edge('notification_emitter',   'cap_radio_difussion')
    graph.add_edge('notification_emitter',   'notification_push')
    graph.add_edge('notification_emitter',   'delivery_state_tracker')
    graph.add_edge('delivery_state_tracker', 'delivery_state_db')

    return components, graph


def simulate_atypical_resilient(transaction, components, metrics, dlq):
    """
    Táctica: datos atípicos detectados en data_validator (temprano).
    System 1 reintenta con datos corregidos (MAX_RETRIES).
    Circuit Breaker actúa si la fuente supera el umbral de fallos consecutivos.
    Transacciones sin salida van a la Dead Letter Queue.
    """
    if transaction.transaction_type == 'alert_trigger':
        channel_comp = transaction.channel if transaction.channel in components else 'sms_service'
        components_sequence = [
            'api_gateway', 'data_validator', 'event_bus',
            'alert_rules_engine_ms', 'alerts_events_db',
            'notification_emitter', channel_comp
        ]
    elif transaction.transaction_type == 'status_query':
        components_sequence = ['api_gateway', 'event_bus', 'delivery_state_tracker', 'delivery_state_db']
    else:
        components_sequence = []

    transaction.components_involved = components_sequence

    for attempt in range(MAX_RETRIES + 1):
        success = True
        validation_rejected = False

        for comp_name in components_sequence:
            comp = components.get(comp_name)
            if not comp or not comp.process_transaction(transaction):
                success = False
                if hasattr(transaction, 'validation_error'):
                    validation_rejected = True
                break

        if success:
            transaction.status = 'Success'
            transaction.retry_count = attempt
            break
        elif validation_rejected and not getattr(transaction, 'circuit_open', False) and attempt < MAX_RETRIES:
            transaction.is_atypical = False
            del transaction.validation_error
        else:
            transaction.status = 'Failed'
            transaction.retry_count = attempt
            dlq.append({
                'source'  : transaction.source,
                'error'   : getattr(transaction, 'validation_error', 'capacity exceeded'),
                'retries' : attempt,
                'circuit' : getattr(transaction, 'circuit_open', False),
            })
            break

    metrics.append(transaction)


def run_resilient_atypical_scenario():
    print(f"=== Táctica: Validación Reforzada + Circuit Breaker + DLQ (tasa atípica={ATYPICAL_DATA_RATE*100:.0f}%) ===")
    print("Tácticas activas:")
    print("  1. ValidationComponent     — detección temprana en data_validator (antes del bus)")
    print(f"  2. Circuit Breaker         — abre tras {CIRCUIT_BREAKER_THRESHOLD} rechazos consecutivos ({CIRCUIT_BREAKER_TIMEOUT}s timeout)")
    print("  3. Retry con datos limpios — System 1 corrige y reintenta (MAX_RETRIES=3)")
    print("  4. Dead Letter Queue       — cuarentena para transacciones sin salida\n")

    components, graph = setup_architecture_resilient_atypical()
    metrics  = []
    dlq      = []
    channels = ['sms_service', 'cap_radio_difussion', 'notification_push']
    threads  = []

    for i in range(1, 1201):
        source = 'data_adquisition_system' if i % 2 == 0 else 'c4_system'
        tx = Transaction(source, 'alert_trigger', random.choice(channels))
        tx.is_atypical  = (source == 'data_adquisition_system') and (random.random() < ATYPICAL_DATA_RATE)
        tx.retry_count  = 0
        tx.circuit_open = False
        t = threading.Thread(target=simulate_atypical_resilient, args=(tx, components, metrics, dlq))
        threads.append(t)
        t.start()

    for i in range(1, 801):
        tx = Transaction('personnel_orchestration_system', 'status_query')
        tx.is_atypical  = False
        tx.retry_count  = 0
        tx.circuit_open = False
        t = threading.Thread(target=simulate_atypical_resilient, args=(tx, components, metrics, dlq))
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    total     = len(metrics)
    success   = sum(1 for tx in metrics if tx.status == 'Success')
    failed    = sum(1 for tx in metrics if tx.status == 'Failed')
    retried   = sum(1 for tx in metrics if getattr(tx, 'retry_count', 0) > 0)
    recovered = sum(1 for tx in metrics if getattr(tx, 'retry_count', 0) > 0 and tx.status == 'Success')
    cb_blocks = sum(1 for e in dlq if e['circuit'])

    print(f"=== Resultados ===")
    print(f"Total Transacciones : {total}")
    print(f"Exitosas            : {success}")
    print(f"Fallidas            : {failed}")
    print(f"\n--- Circuit Breaker ---")
    print(f"Rechazos rápidos por CB                : {cb_blocks}")
    print(f"\n--- Retry (System 1 corrige datos) ---")
    print(f"Requirieron reintento                  : {retried}")
    print(f"Recuperados tras retry                 : {recovered}")
    print(f"\n--- Dead Letter Queue ---")
    print(f"Entradas en cuarentena                 : {len(dlq)}")


run_resilient_atypical_scenario()

=== Táctica: Validación Reforzada + Circuit Breaker + DLQ (tasa atípica=30%) ===
Tácticas activas:
  1. ValidationComponent     — detección temprana en data_validator (antes del bus)
  2. Circuit Breaker         — abre tras 5 rechazos consecutivos (2.0s timeout)
  3. Retry con datos limpios — System 1 corrige y reintenta (MAX_RETRIES=3)
  4. Dead Letter Queue       — cuarentena para transacciones sin salida

=== Resultados ===
Total Transacciones : 2000
Exitosas            : 1408
Fallidas            : 592

--- Circuit Breaker ---
Rechazos rápidos por CB                : 0

--- Retry (System 1 corrige datos) ---
Requirieron reintento                  : 122
Recuperados tras retry                 : 89

--- Dead Letter Queue ---
Entradas en cuarentena                 : 592


# Escenario de Calidad: Idempotencia

El sistema 1 envía datos duplicados, lecturas con el mismo timestamp y lecturas fuera de orden con gaps intercalados.
El `alert_rules_engine_ms` actúa como pipeline de idempotencia antes de procesar los eventos.

| Atributo | Descripción |
|---|---|
| **Estímulo** | `data_adquisition_system` envía duplicados, timestamps repetidos y secuencias desordenadas con gaps |
| **Fuente** | Sistema 1 (`data_adquisition_system`) |
| **Entorno** | Operación normal; 20% duplicados · 15% fuera de orden · 10% prob. de gap por transacción |
| **Artefacto** | `alert_rules_engine_ms` (pipeline de idempotencia) + `delivery_state_tracker` (auditoría de fallos) |
| **Respuesta** | El motor de reglas filtra duplicados, normaliza timestamps, reordena y detecta gaps; el rastreador de entrega captura todos los casos fallidos del pipeline |
| **Medida** | Duplicados filtrados, eventos fuera de orden detectados, gaps encontrados, entradas de auditoría generadas |

**Atributos de calidad**: Fiabilidad · Consistencia · Integridad · Interoperabilidad · Capacidad de auditoría

In [30]:
# ================================
# Escenario de Calidad — Idempotencia (Arquitectura Original)
# ================================
import hashlib

DUPLICATE_RATE    = 0.20  # 20% de alertas son duplicados exactos
OUT_OF_ORDER_RATE = 0.15  # 15% llegan con sequence_num fuera de orden
GAP_PROBABILITY   = 0.10  # 10% de prob. de gap en la secuencia por transacción


def setup_architecture_idempotency_original():
    """Arquitectura original — alert_rules_engine_ms sin pipeline de idempotencia."""
    components = {}
    graph = nx.DiGraph()

    components['api_gateway']            = PresentationTier('api_gateway', capacity=200)
    components['data_validator']          = LogicTier('data_validator', capacity=200)
    components['event_bus']              = LogicTier('event_bus', capacity=500)
    components['alert_rules_engine_ms']  = LogicTier('alert_rules_engine_ms', capacity=300)  # Sin idempotencia
    components['notification_emitter']   = LogicTier('notification_emitter', capacity=300)
    components['delivery_state_tracker'] = LogicTier('delivery_state_tracker', capacity=200)
    components['sms_service']            = ExternalServiceTier('sms_service', capacity=500)
    components['cap_radio_difussion']    = ExternalServiceTier('cap_radio_difussion', capacity=300)
    components['notification_push']      = ExternalServiceTier('notification_push', capacity=400)
    components['alerts_events_db']       = DataTier('alerts_events_db', capacity=300)
    components['delivery_state_db']      = DataTier('delivery_state_db', capacity=300)
    components['alerts_events_cache']    = DataTier('alerts_events_cache', capacity=400)

    graph.add_edge('api_gateway',           'data_validator')
    graph.add_edge('data_validator',         'event_bus')
    graph.add_edge('event_bus',              'alert_rules_engine_ms')
    graph.add_edge('event_bus',              'notification_emitter')
    graph.add_edge('event_bus',              'delivery_state_tracker')
    graph.add_edge('alert_rules_engine_ms',  'alerts_events_db')
    graph.add_edge('alert_rules_engine_ms',  'alerts_events_cache')
    graph.add_edge('notification_emitter',   'sms_service')
    graph.add_edge('notification_emitter',   'cap_radio_difussion')
    graph.add_edge('notification_emitter',   'notification_push')
    graph.add_edge('notification_emitter',   'delivery_state_tracker')
    graph.add_edge('delivery_state_tracker', 'delivery_state_db')

    return components, graph


def simulate_idempotency_original(transaction, components, metrics):
    """Flujo original: duplicados procesados, fuera de orden sin reordenar, gaps sin detectar."""
    if transaction.transaction_type == 'alert_trigger':
        channel_comp = transaction.channel if transaction.channel in components else 'sms_service'
        components_sequence = [
            'api_gateway', 'data_validator', 'event_bus',
            'alert_rules_engine_ms', 'alerts_events_db',
            'notification_emitter', channel_comp
        ]
    elif transaction.transaction_type == 'status_query':
        components_sequence = ['api_gateway', 'event_bus', 'delivery_state_tracker', 'delivery_state_db']
    else:
        components_sequence = []

    transaction.components_involved = components_sequence
    success = True
    for comp_name in components_sequence:
        comp = components.get(comp_name)
        if not comp or not comp.process_transaction(transaction):
            success = False
            break
    transaction.status = 'Success' if success else 'Failed'
    metrics.append(transaction)


def run_idempotency_original_scenario():
    print("=== Escenario de Calidad: Idempotencia — Arquitectura Original ===")
    print(f"  Duplicados     : {DUPLICATE_RATE*100:.0f}% de alertas")
    print(f"  Fuera de orden : {OUT_OF_ORDER_RATE*100:.0f}% de alertas")
    print(f"  Prob. de gap   : {GAP_PROBABILITY*100:.0f}% por transacción")
    print("  alert_rules_engine_ms NO tiene pipeline de idempotencia\n")

    components, graph = setup_architecture_idempotency_original()
    metrics  = []
    channels = ['sms_service', 'cap_radio_difussion', 'notification_push']
    threads  = []

    base_ts  = time.time()
    seq_ctrs = defaultdict(int)
    tx_pool  = []

    for i in range(1, 1201):
        source = 'data_adquisition_system' if i % 2 == 0 else 'c4_system'

        seq = seq_ctrs[source]
        if random.random() < GAP_PROBABILITY:
            seq += random.randint(2, 4)
        seq_ctrs[source] = seq + 1

        tx = Transaction(source, 'alert_trigger', random.choice(channels))
        tx.is_duplicate = False
        tx.out_of_order = False
        tx.gap_size     = 0

        if random.random() < DUPLICATE_RATE and tx_pool:
            ref = random.choice(tx_pool[-30:])
            tx.timestamp    = ref.timestamp
            tx.sequence_num = ref.sequence_num
            tx.is_duplicate = True
        elif random.random() < OUT_OF_ORDER_RATE and seq > 0:
            tx.sequence_num = max(0, seq - random.randint(1, 3))
            tx.timestamp    = base_ts + tx.sequence_num * 0.05
            tx.out_of_order = True
        else:
            tx.sequence_num = seq
            tx.timestamp    = base_ts + seq * 0.05
            if seq > seq_ctrs[source] - 2:
                tx.gap_size = seq - (seq_ctrs[source] - 2) if seq > (seq_ctrs[source] - 2) else 0

        tx_pool.append(tx)
        t = threading.Thread(target=simulate_idempotency_original, args=(tx, components, metrics))
        threads.append(t)
        t.start()

    for i in range(1, 801):
        tx = Transaction('personnel_orchestration_system', 'status_query')
        t = threading.Thread(target=simulate_idempotency_original, args=(tx, components, metrics))
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    total    = len(metrics)
    success  = sum(1 for tx in metrics if tx.status == 'Success')
    failed   = sum(1 for tx in metrics if tx.status == 'Failed')
    dup_notif    = sum(1 for tx in metrics if getattr(tx, 'is_duplicate', False) and tx.status == 'Success')
    ooo_notif    = sum(1 for tx in metrics if getattr(tx, 'out_of_order', False) and tx.status == 'Success')
    gaps_ignored = sum(1 for tx in metrics if getattr(tx, 'gap_size', 0) > 0 and tx.status == 'Success')

    print(f"=== Resultados ===")
    print(f"Total Transacciones : {total}")
    print(f"Exitosas            : {success}")
    print(f"Fallidas            : {failed}")
    print(f"\n--- Impacto sin Pipeline de Idempotencia ---")
    print(f"Notificaciones duplicadas enviadas          : {dup_notif}  ← violación de integridad")
    print(f"Eventos fuera de orden procesados sin reord.: {ooo_notif}  ← violación de consistencia")
    print(f"Gaps no detectados (datos faltantes)        : {gaps_ignored}  ← violación de completitud")


run_idempotency_original_scenario()

=== Escenario de Calidad: Idempotencia — Arquitectura Original ===
  Duplicados     : 20% de alertas
  Fuera de orden : 15% de alertas
  Prob. de gap   : 10% por transacción
  alert_rules_engine_ms NO tiene pipeline de idempotencia

=== Resultados ===
Total Transacciones : 2000
Exitosas            : 1421
Fallidas            : 579

--- Impacto sin Pipeline de Idempotencia ---
Notificaciones duplicadas enviadas          : 164  ← violación de integridad
Eventos fuera de orden procesados sin reord.: 102  ← violación de consistencia
Gaps no detectados (datos faltantes)        : 492  ← violación de completitud


# Táctica Arquitectónica: Resiliencia ante Datos Duplicados y Desordenados

El `alert_rules_engine_ms` incorpora un pipeline de idempotencia con las siguientes acciones:

| Táctica | Descripción |
|---|---|
| **Generación de clave de idempotencia** | Clave única `MD5(source:timestamp:sequence_num)` por evento — identifica de forma determinista cualquier dato duplicado |
| **Filtro de duplicados** | Set de claves ya procesadas; si la clave existe, el evento se rechaza inmediatamente sin reprocessing |
| **Normalización de timestamps** | Los timestamps se redondean a segundo entero, eliminando variaciones de precisión que generarían falsos duplicados |
| **Reordenamiento y detección de gaps** | Seguimiento de `expected_seq` por fuente: detecta eventos fuera de orden y gaps en la secuencia |
| **Rastreador de entrega (auditoría)** | `delivery_state_tracker` gestiona todos los casos fallidos del pipeline del motor de reglas, garantizando trazabilidad completa |

> **Hipótesis de mejora**: los duplicados se eliminan antes de generar notificaciones, los eventos fuera de orden se detectan y marcan, los gaps quedan registrados en auditoría. El sistema pasa de procesar silenciosamente datos incorrectos a detectarlos y reportarlos.

In [31]:
# ================================
# Táctica: Pipeline de Idempotencia en alert_rules_engine_ms
# ================================


class IdempotentRulesEngine(Component):
    """
    alert_rules_engine_ms con pipeline de idempotencia:
    1. Generación de clave de idempotencia  — MD5(source:timestamp:sequence_num)
    2. Filtro de duplicados                 — set de claves ya procesadas
    3. Normalización de timestamps          — redondeo a segundo entero
    4. Reordenamiento y detección de gaps   — expected_seq por fuente
    """
    def __init__(self, name, capacity=300):
        super().__init__(name, 'Logic Tier', capacity)
        self._seen_keys    = set()
        self._keys_lock    = threading.Lock()
        self._expected_seq = defaultdict(int)
        self._seq_lock     = threading.Lock()
        self.stats         = defaultdict(int)

    def _idempotency_key(self, tx):
        raw = f"{tx.source}:{getattr(tx,'timestamp',0):.3f}:{getattr(tx,'sequence_num',0)}"
        return hashlib.md5(raw.encode()).hexdigest()

    def _normalize_timestamp(self, tx):
        if hasattr(tx, 'timestamp'):
            tx.timestamp_normalized = round(tx.timestamp)
            self.stats['timestamps_normalized'] += 1

    def _check_sequence(self, tx):
        src = tx.source
        seq = getattr(tx, 'sequence_num', None)
        if seq is None:
            return
        with self._seq_lock:
            expected = self._expected_seq[src]
            if seq < expected:
                tx.out_of_order = True
                self.stats['out_of_order'] += 1
            elif seq > expected:
                tx.gap_size = seq - expected
                self.stats['gaps_detected']  += 1
                self.stats['missing_events'] += tx.gap_size
            self._expected_seq[src] = max(self._expected_seq[src], seq + 1)

    def process_transaction(self, tx):
        with self.lock:
            if self.current_load >= self.capacity:
                return False
            self.current_load += 1

        time.sleep(random.uniform(0.01, 0.05))

        # Táctica 1: generar clave de idempotencia
        key = self._idempotency_key(tx)
        tx.idempotency_key = key

        # Táctica 2: filtro de duplicados
        with self._keys_lock:
            is_dup = key in self._seen_keys
            if not is_dup:
                self._seen_keys.add(key)

        if is_dup:
            with self.lock:
                self.current_load -= 1
            tx.pipeline_error = 'DUPLICATE: clave de idempotencia ya procesada'
            self.stats['duplicates_filtered'] += 1
            return False

        # Táctica 3: normalización de timestamps
        self._normalize_timestamp(tx)

        # Táctica 4: reordenamiento y detección de gaps
        self._check_sequence(tx)

        with self.lock:
            self.current_load -= 1
        return True


def setup_architecture_idempotency_resilient():
    """Arquitectura modificada: alert_rules_engine_ms con pipeline de idempotencia."""
    components = {}
    graph = nx.DiGraph()

    components['api_gateway']            = PresentationTier('api_gateway', capacity=200)
    components['data_validator']          = LogicTier('data_validator', capacity=200)
    components['event_bus']              = LogicTier('event_bus', capacity=500)
    components['alert_rules_engine_ms']  = IdempotentRulesEngine('alert_rules_engine_ms', capacity=300)
    components['notification_emitter']   = LogicTier('notification_emitter', capacity=300)
    components['delivery_state_tracker'] = LogicTier('delivery_state_tracker', capacity=200)
    components['sms_service']            = ExternalServiceTier('sms_service', capacity=500)
    components['cap_radio_difussion']    = ExternalServiceTier('cap_radio_difussion', capacity=300)
    components['notification_push']      = ExternalServiceTier('notification_push', capacity=400)
    components['alerts_events_db']       = DataTier('alerts_events_db', capacity=300)
    components['delivery_state_db']      = DataTier('delivery_state_db', capacity=300)
    components['alerts_events_cache']    = DataTier('alerts_events_cache', capacity=400)

    graph.add_edge('api_gateway',           'data_validator')
    graph.add_edge('data_validator',         'event_bus')
    graph.add_edge('event_bus',              'alert_rules_engine_ms')
    graph.add_edge('event_bus',              'notification_emitter')
    graph.add_edge('event_bus',              'delivery_state_tracker')
    graph.add_edge('alert_rules_engine_ms',  'alerts_events_db')
    graph.add_edge('alert_rules_engine_ms',  'alerts_events_cache')
    graph.add_edge('notification_emitter',   'sms_service')
    graph.add_edge('notification_emitter',   'cap_radio_difussion')
    graph.add_edge('notification_emitter',   'notification_push')
    graph.add_edge('notification_emitter',   'delivery_state_tracker')
    graph.add_edge('delivery_state_tracker', 'delivery_state_db')

    return components, graph


def simulate_idempotency_resilient(transaction, components, metrics, audit_log):
    """
    Flujo con táctica:
    - Si alert_rules_engine_ms rechaza (duplicado): delivery_state_tracker
      registra el caso fallido del pipeline (táctica 5 — auditoría).
    - Eventos out-of-order o con gap: procesados y marcados en stats del motor.
    """
    if transaction.transaction_type == 'status_query':
        seq = ['api_gateway', 'event_bus', 'delivery_state_tracker', 'delivery_state_db']
        transaction.components_involved = seq
        success = True
        for n in seq:
            comp = components.get(n)
            if not comp or not comp.process_transaction(transaction):
                success = False
                break
        transaction.status = 'Success' if success else 'Failed'
        metrics.append(transaction)
        return

    channel_comp = transaction.channel if transaction.channel in components else 'sms_service'
    pre_rules  = ['api_gateway', 'data_validator', 'event_bus']
    post_rules = ['alerts_events_db', 'notification_emitter', channel_comp]
    transaction.components_involved = pre_rules + ['alert_rules_engine_ms'] + post_rules

    success = True
    for comp_name in pre_rules:
        comp = components.get(comp_name)
        if not comp or not comp.process_transaction(transaction):
            success = False
            break

    if not success:
        transaction.status = 'Failed'
        metrics.append(transaction)
        return

    rules_comp = components.get('alert_rules_engine_ms')
    rules_ok   = rules_comp and rules_comp.process_transaction(transaction)

    if not rules_ok:
        # Táctica 5: delivery_state_tracker gestiona los casos fallidos del pipeline
        tracker = components.get('delivery_state_tracker')
        if tracker:
            tracker.process_transaction(transaction)
        audit_log.append({
            'source'          : transaction.source,
            'idempotency_key' : getattr(transaction, 'idempotency_key', 'N/A'),
            'error'           : getattr(transaction, 'pipeline_error', 'rules engine failure'),
            'out_of_order'    : getattr(transaction, 'out_of_order', False),
            'gap_size'        : getattr(transaction, 'gap_size', 0),
        })
        transaction.status = 'Filtered'
        metrics.append(transaction)
        return

    for comp_name in post_rules:
        comp = components.get(comp_name)
        if not comp or not comp.process_transaction(transaction):
            success = False
            break

    transaction.status = 'Success' if success else 'Failed'
    metrics.append(transaction)


def run_resilient_idempotency_scenario():
    print("=== Táctica: Pipeline de Idempotencia en alert_rules_engine_ms ===")
    print("Tácticas activas:")
    print("  1. Generación de clave de idempotencia  — MD5(source:timestamp:sequence_num)")
    print("  2. Filtro de duplicados                 — set de claves ya procesadas")
    print("  3. Normalización de timestamps          — redondeo a segundo entero")
    print("  4. Reordenamiento y detección de gaps   — expected_seq por fuente")
    print("  5. delivery_state_tracker               — gestión de casos fallidos del pipeline\n")

    components, graph = setup_architecture_idempotency_resilient()
    metrics   = []
    audit_log = []
    channels  = ['sms_service', 'cap_radio_difussion', 'notification_push']
    threads   = []

    base_ts  = time.time()
    seq_ctrs = defaultdict(int)
    tx_pool  = []

    for i in range(1, 1201):
        source = 'data_adquisition_system' if i % 2 == 0 else 'c4_system'

        seq = seq_ctrs[source]
        if random.random() < GAP_PROBABILITY:
            seq += random.randint(2, 4)
        seq_ctrs[source] = seq + 1

        tx = Transaction(source, 'alert_trigger', random.choice(channels))
        tx.out_of_order = False
        tx.gap_size     = 0

        if random.random() < DUPLICATE_RATE and tx_pool:
            ref = random.choice(tx_pool[-30:])
            tx.timestamp    = ref.timestamp
            tx.sequence_num = ref.sequence_num
        elif random.random() < OUT_OF_ORDER_RATE and seq > 0:
            tx.sequence_num = max(0, seq - random.randint(1, 3))
            tx.timestamp    = base_ts + tx.sequence_num * 0.05
        else:
            tx.sequence_num = seq
            tx.timestamp    = base_ts + seq * 0.05

        tx_pool.append(tx)
        t = threading.Thread(
            target=simulate_idempotency_resilient,
            args=(tx, components, metrics, audit_log)
        )
        threads.append(t)
        t.start()

    for i in range(1, 801):
        tx = Transaction('personnel_orchestration_system', 'status_query')
        t = threading.Thread(
            target=simulate_idempotency_resilient,
            args=(tx, components, metrics, audit_log)
        )
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    s        = components['alert_rules_engine_ms'].stats
    total    = len(metrics)
    success  = sum(1 for tx in metrics if tx.status == 'Success')
    filtered = sum(1 for tx in metrics if tx.status == 'Filtered')
    failed   = sum(1 for tx in metrics if tx.status == 'Failed')

    print(f"=== Resultados ===")
    print(f"Total transacciones     : {total}")
    print(f"Exitosas                : {success}")
    print(f"Filtradas (intencional) : {filtered}")
    print(f"Fallidas                : {failed}")
    print(f"\n--- Pipeline de Idempotencia (alert_rules_engine_ms) ---")
    print(f"Duplicados filtrados        : {s['duplicates_filtered']}")
    print(f"Timestamps normalizados     : {s['timestamps_normalized']}")
    print(f"Eventos fuera de orden      : {s['out_of_order']}")
    print(f"Gaps detectados             : {s['gaps_detected']} ({s['missing_events']} eventos faltantes)")
    print(f"\n--- Táctica 5: delivery_state_tracker (auditoría de pipeline) ---")
    print(f"Casos fallidos registrados  : {len(audit_log)}")


run_resilient_idempotency_scenario()


=== Táctica: Pipeline de Idempotencia con TTL + Reordenamiento + Compensación ===
Tácticas activas:
  1. Idempotencia con TTL      — claves expiran tras 60s (memoria acotada)
  2. Buffer de reordenamiento  — min-heap por fuente, tamaño máx 10
  3. Compensación de gaps      — recovery requests para eventos faltantes
  4. Auditoría via delivery_state_tracker — registro de todos los casos filtrados

=== Resultados ===
Total transacciones     : 2000
Exitosas                : 1390
Filtradas (intencional) : 165
Fallidas                : 445

--- Pipeline de Idempotencia ---
Duplicados filtrados          : 165
Claves expiradas (TTL)        : 0
Timestamps normalizados       : 646
Eventos fuera de orden        : 614
Liberados del buffer (reord.) : 594
Gaps detectados               : 32 (1581 eventos faltantes)
Solicitudes de compensación   : 1581 eventos a recuperar

--- Auditoría (delivery_state_tracker) ---
Entradas de auditoría         : 165
